
# Feature Engineering

If data is the fuel for machine learning, **Feature Engineering** is the refinery. It is the process of using domain knowledge to extract new variables from raw data that make machine learning algorithms work better.

While algorithms (like Linear Regression or Random Forest) are mathematical commodities, feature engineering is an **art**. It is often the deciding factor between a mediocre model and a winning one.

## Why do we need it?

Models typically assume a linear relationship or have difficulty seeing complex patterns.

-   **Example**: A model might not understand that "Total Bill" and "Tip" are related to "Percentage." If you create a new feature `Tip_Percentage = Tip / Total Bill`, you make the pattern obvious to the model.

## Common Techniques

| Technique            | Description                                           | Example                                   |
|-------------------- |----------------------------------------------------- |----------------------------------------- |
| **Transformation**   | Changing the scale or distribution of a feature.      | Logarithm (for skewed data), Square Root. |
| **Interaction**      | Combining two features to capture their joint effect. | `Income` $\times$ `Education`.            |
| **Ratio/Derivation** | Creating rates or densities.                          | `Price` / `SquareFoot` = `PricePerSqFt`.  |
| **Polynomials**      | Adding curvature to linear models.                    | $x^2$, $x^3$ allowing the line to curve.  |
| **Binning**          | Grouping continuous numbers into categories.          | Age 25 $\rightarrow$ "Young Adult".       |

## Practical Demonstration: California Housing

We will demonstrate how adding a few simple features can improve model performance on the California Housing dataset.

### Establish a Baseline

First, we train a model on the raw data to get a "score to beat."

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load Data
data = fetch_california_housing(as_frame=True)
df = data.frame.copy() # Use copy to avoid pandas warnings later

# Split
X = df.drop(columns=[data.target.name])
y = df[data.target.name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline Model
base_model = LinearRegression()
base_model.fit(X_train, y_train)
base_score = base_model.score(X_test, y_test)

print(f"Baseline R² Score: {base_score:.4f}")

### Engineering New Features

Now we apply our domain intuition.

1.  **Ratio**: Total rooms in a block isn't useful. Rooms **per person** is useful.
2.  **Polynomial**: Income often has a non-linear relationship with house price (diminishing returns). We add $Income^2$.

In [ ]:
# 1. Create Ratio Feature: Rooms per Person
df['Rooms_per_person'] = df['AveRooms'] / df['AveOccup']

# 2. Create Polynomial Feature: Income Squared
df['MedInc_Squared'] = df['MedInc'] ** 2

print(df[['MedInc', 'MedInc_Squared', 'AveRooms', 'Rooms_per_person']].head())

**Note**: Adding new variables that could be correlated with existing ones can lead to *multicolinearity*, to be discussed later.

### Evaluate Improvement

Let's retrain the model with these new features included.

In [ ]:
# Prepare new X
X_eng = df.drop(columns=[data.target.name])
y_eng = df[data.target.name]

# Split (same random_state ensures fair comparison)
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42)

# Train New Model
eng_model = LinearRegression()
eng_model.fit(X_train_e, y_train_e)
eng_score = eng_model.score(X_test_e, y_test_e)

print(f"Baseline R²:   {base_score:.4f}")
print(f"Engineered R²: {eng_score:.4f}")
print(f"Improvement:   {(eng_score - base_score) * 100:.2f}%")

**Result**: By simply adding two logical features, we slightly improved the model's predictive power.

## Exercises

Now it's your turn to squeeze more performance out of this dataset.

### Create a Bedroom Ratio

The raw dataset has `AveBedrms` (Average Bedrooms) and `AveRooms` (Average Rooms).

-   Create a feature `Bedrooms_Ratio` = `AveBedrms` / `AveRooms`.
-   Logic: This acts as a proxy for "Apartment vs. House". High ratio might mean small apartments.

### Log Transformation

Income distributions are often "skewed" (a few very rich people, many average). Linear models prefer "Normal" (bell-curve) distributions.

-   Create `Log_Income` = `log(MedInc + 1)`.
-   **Note**: We use `np.log1p(x)` which calculates `log(1+x)` accurately.

**Note**: If the correlation increases, the linear relationship is stronger!

### Final Test

Train a final model with all your new features combined.

## Summary

Feature Engineering is about translating real-world logic into math.

1.  We created **Interaction/Ratio** features (`Rooms_per_person`).
2.  We used **Mathematical Transformations** (`Log`, `Square`).
3.  We showed that **Better Data > Better Algorithms**. A Linear Regression with good features can often beat a complex Neural Network with raw data.